# 🪖 가상 주식 애널리스트 실습

**국방 소프트웨어 관리** | AI를 활용한 다중 소스 데이터 융합과 보고서 생성

## 학습 목표
- 외부 데이터(시세, 뉴스, 기업정보)와 AI를 결합한 분석 도구 구현
- AI 출력의 검증 가능성 확보 (근거 표기 강제)
- 할루시네이션 위험과 비결정성을 직접 체감
- 자격증명(API 키)의 안전 관리 실습

## 오늘의 과제
종목 코드를 입력하면 AI가 데이터 기반 투자 분석 보고서를 생성하는 Gradio 웹앱 제작

---

⚠️ **본 실습은 교육용입니다. 작성된 보고서는 실제 투자 자문이 아닙니다.**

## 🔑 사전 준비 (실습 시작 전 필수)

### 1단계: Anthropic 계정 생성
1. https://console.anthropic.com 접속
2. 이메일로 가입 → 전화번호 인증
3. 가입 시 약 $5 스타터 크레딧 자동 지급 (오늘 실습에 충분)

### 2단계: API 키 발급
1. Console → **Settings → API Keys**
2. **Create Key** 클릭 → 이름 입력 (예: `defense-sw-class`)
3. 표시된 키(`sk-ant-...`)를 **즉시 복사** (한 번만 표시됩니다)

### 3단계: Colab 보안 비밀에 저장
1. Colab 좌측 사이드바의 🔑 (열쇠) 아이콘 클릭
2. **새 보안 비밀** 추가
3. 이름: `ANTHROPIC_API_KEY`
4. 값: 복사한 키 붙여넣기
5. **노트북 액세스** 토글을 **ON**

### 4단계: 사용 한도 설정 (권장)
Console → **Settings → Limits**에서 월 지출 한도를 `$5` 정도로 설정.
키가 유출되어도 그 이상 과금되지 않습니다.

---

⚠️ **절대 코드 셀에 API 키를 직접 입력하지 마세요.**

> **국방 SW 관리 관점**: API 키는 자격증명입니다. 코드와 함께 유출되면
> 외부에서 해당 계정의 모든 권한을 사용할 수 있습니다. 보안 저장소와
> 사용 한도는 함께 갖춰야 하는 기본 통제입니다.

## 📦 환경 설정

In [ ]:
!pip install -q yfinance anthropic gradio

In [ ]:
import os
from google.colab import userdata

# Colab 보안 비밀에서 API 키 로드 (코드에 키가 노출되지 않음)
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from anthropic import Anthropic

client = Anthropic()
print("✅ 준비 완료")

## 📊 1단계: 종목 데이터 가져오기

`yfinance` 라이브러리로 Yahoo Finance에서 데이터를 받아옵니다.

**종목 코드 형식**
- 한국 주식: `종목번호.KS` (예: 삼성전자 `005930.KS`)
- 미국 주식: 그대로 (예: `AAPL`, `LMT`)

**방산주 예시 (이 실습 추천)**

| 회사 | 코드 |
|---|---|
| 한화에어로스페이스 | `012450.KS` |
| LIG넥스원 | `079550.KS` |
| 한국항공우주 | `047810.KS` |
| 현대로템 | `064350.KS` |
| Lockheed Martin (미국) | `LMT` |
| RTX (구 Raytheon) | `RTX` |

In [ ]:
TICKER = "012450.KS"  # 한화에어로스페이스 (자유롭게 변경하세요)

stock = yf.Ticker(TICKER)
info = stock.info

print("기업명:", info.get("longName"))
print("산업:", info.get("industry"))
print("국가:", info.get("country"))
print("\n사업 개요:")
print(info.get("longBusinessSummary", "정보 없음")[:300], "...")

## 📈 2단계: 주가 차트와 핵심 지표

최근 3개월치 종가 데이터로 차트와 요약 지표를 출력합니다.
**여기까지는 AI를 전혀 사용하지 않은 전통적 데이터 분석입니다.**

In [ ]:
hist = stock.history(period="3mo")

# 차트
plt.figure(figsize=(10, 4))
hist["Close"].plot(title=f"{info.get('longName')} 최근 3개월 종가")
plt.ylabel("가격")
plt.grid(alpha=0.3)
plt.show()

# 핵심 지표
current = hist["Close"].iloc[-1]
change_pct = (current / hist["Close"].iloc[0] - 1) * 100
avg_vol = hist["Volume"].mean()

print(f"현재가:        {current:>12,.0f}")
print(f"3개월 등락률:  {change_pct:>+11.1f}%")
print(f"평균 거래량:   {avg_vol:>12,.0f}")

## 📰 3단계: 최근 뉴스 헤드라인

`yfinance`는 종목 관련 뉴스도 함께 제공합니다.
라이브러리 버전에 따라 응답 구조가 달라질 수 있어 **방어 코드**를 함께 작성합니다.
(외부 API에 의존할 때의 일반 원칙)

In [ ]:
news_titles = []
try:
    for n in stock.news[:5]:
        # 버전별 응답 구조 차이를 흡수
        title = n.get("title") or n.get("content", {}).get("title", "")
        if title:
            news_titles.append(title)
except Exception as e:
    print("뉴스 수집 실패:", e)

for i, t in enumerate(news_titles, 1):
    print(f"{i}. {t}")

if not news_titles:
    print("(가져온 뉴스가 없습니다 — 다른 종목으로 시도해 보세요)")

## 🤖 4단계: Claude로 보고서 생성 (실습의 핵심)

지금까지 수집한 데이터를 Claude에게 넘겨 분석 보고서를 작성하도록 합니다.

### 핵심 설계 포인트

이 프롬프트의 가장 중요한 부분은 **`[근거: ...]` 강제 표기**입니다.

- AI가 어떤 데이터를 보고 결론을 내렸는지 추적 가능해야 검증 가능
- 군 정보판단에서 "출처(source)"와 "신뢰도(reliability)"를 명시하는 원칙과 동일
- 근거 없는 일반론을 차단하여 **할루시네이션 위험 감소**

In [ ]:
price_summary = hist["Close"].describe().to_string()
news_text = "\n".join(f"- {t}" for t in news_titles) or "(뉴스 없음)"

prompt = f"""당신은 신중한 주식 애널리스트입니다.
아래 데이터만을 근거로 한국어 보고서를 작성하세요.
데이터에 없는 정보는 절대 추측하지 마세요.

== 기업 개요 ==
{info.get('longBusinessSummary', '')}

== 최근 3개월 주가 통계 ==
{price_summary}

== 최근 뉴스 헤드라인 ==
{news_text}

다음 형식으로 작성하세요:
1. 종목 개요 (2~3문장)
2. 최근 주가 동향
3. 강점 3가지
4. 리스크 3가지
5. 종합 의견 (매수/중립/매도 + 근거)

⚠️ 모든 주장 끝에 반드시 [근거: 위 데이터 중 어느 부분] 형식으로 출처를 명시하세요.
근거가 없는 일반론은 작성하지 마세요."""

response = client.messages.create(
    model="claude-sonnet-4-6",  # 비용 절감용. 더 좋은 품질은 "claude-opus-4-7"
    max_tokens=2000,
    messages=[{"role": "user", "content": prompt}]
)

report = response.content[0].text
print(report)

## 🖥️ 5단계: Gradio로 통합 웹앱 만들기

지금까지의 모든 단계를 하나의 함수로 묶고 Gradio 웹 인터페이스를 입힙니다.

**Gradio의 장점**
- 코드 몇 줄로 웹 UI 생성
- Colab에서 `share=True` 옵션으로 72시간 공유 가능한 URL 자동 생성
- 다른 학생이나 지휘부에 데모할 때 유용

**검증용 패널 포함**
"🔍 AI가 본 원본 데이터" 섹션을 펼치면 AI에 전달된 원본 데이터를
직접 확인할 수 있습니다. 이것이 이 앱이 단순 챗봇과 다른 점입니다.

In [ ]:
import gradio as gr

def analyze_stock(ticker_input):
    try:
        stock = yf.Ticker(ticker_input)
        info = stock.info
        hist = stock.history(period="3mo")

        if hist.empty or not info.get("longName"):
            return (
                "❌ 종목을 찾을 수 없습니다. 코드를 확인하세요 (한국 주식은 .KS 필요)",
                None,
                ""
            )

        # 차트
        fig, ax = plt.subplots(figsize=(10, 4))
        hist["Close"].plot(ax=ax, title=info.get("longName"))
        ax.grid(alpha=0.3)
        ax.set_ylabel("가격")

        # 뉴스
        news_titles = []
        try:
            for n in stock.news[:5]:
                title = n.get("title") or n.get("content", {}).get("title", "")
                if title:
                    news_titles.append(title)
        except Exception:
            pass

        # 원본 데이터 (검증용)
        price_summary = hist["Close"].describe().to_string()
        news_text = "\n".join(f"- {t}" for t in news_titles) or "(뉴스 없음)"
        raw_data = f"""=== 기업 개요 ===
{info.get('longBusinessSummary', '')}

=== 주가 통계 (최근 3개월) ===
{price_summary}

=== 뉴스 헤드라인 ===
{news_text}"""

        # Claude 호출
        prompt = f"""당신은 신중한 주식 애널리스트입니다.
아래 데이터만을 근거로 한국어 보고서를 작성하세요.
데이터에 없는 정보는 추측하지 마세요.

{raw_data}

다음 형식으로 작성하세요:
1. 종목 개요
2. 최근 주가 동향
3. 강점 3가지
4. 리스크 3가지
5. 종합 의견 (매수/중립/매도 + 근거)

⚠️ 모든 주장 끝에 [근거: ...] 형식으로 출처를 명시하세요."""

        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]
        )
        report = response.content[0].text

        return report, fig, raw_data

    except Exception as e:
        return f"❌ 오류 발생: {e}", None, ""


with gr.Blocks(title="가상 주식 애널리스트") as demo:
    gr.Markdown("# 🪖 가상 주식 애널리스트")
    gr.Markdown("⚠️ 교육용 도구입니다. 실제 투자 자문이 아닙니다.")

    with gr.Row():
        ticker_in = gr.Textbox(
            label="종목 코드",
            value="012450.KS",
            placeholder="예: 012450.KS, AAPL, LMT"
        )
        btn = gr.Button("분석 시작", variant="primary")

    with gr.Row():
        with gr.Column():
            report_out = gr.Markdown(label="AI 분석 보고서")
        with gr.Column():
            chart_out = gr.Plot(label="주가 차트")

    with gr.Accordion("🔍 AI가 본 원본 데이터 (검증용)", open=False):
        raw_out = gr.Textbox(label="", lines=15)

    btn.click(
        analyze_stock,
        inputs=[ticker_in],
        outputs=[report_out, chart_out, raw_out]
    )

demo.launch(share=True, debug=True)

## 🎯 토론 과제 (실습 후반부)

다음 실험을 직접 수행하고 결과를 비교하세요.

### 과제 1: 재현성 검증
같은 종목을 **3번 연속** 분석해 보세요. 결과가 얼마나 일관적인가요?
- 일관되지 않은 부분은 어디인가? (수치? 의견? 표현?)
- 군 의사결정 도구로 사용하려면 어떤 추가 장치가 필요할까?

### 과제 2: 예외 처리
존재하지 않는 종목 코드(예: `XXXXX.KS`)를 입력하면 어떻게 되나요?
- 앱이 깔끔하게 실패하는가, 그럴듯한 거짓을 만들어내는가?

### 과제 3: 근거 검증 (가장 중요)
보고서의 `[근거: ...]` 표기를 **🔍 검증용 패널의 원본 데이터와 직접 대조**하세요.
- AI가 인용한 내용이 실제 데이터와 일치하는가?
- 일치하지 않는 부분이 있다면 그것이 바로 **할루시네이션**입니다.

### 과제 4: 방산주 패턴 비교
다음 종목을 차례로 분석하고 AI 의견의 패턴을 관찰하세요:
- `012450.KS` (한화에어로스페이스)
- `079550.KS` (LIG넥스원)
- `047810.KS` (한국항공우주)

AI가 특정 산업군에 대해 일관된 편향(bias)을 보이는가? 어떤 단어가 반복되는가?

---

## 💭 종합 토론

> **"이 보고서를 그대로 지휘관(또는 투자위원회)에게 제출할 수 있는가?  
> 그렇지 않다면, 무엇을 추가/제거/검증해야 하는가?"**

이 질문이 오늘 실습의 핵심입니다. **AI는 보조 도구이지 결정자가 아닙니다.**

## 📚 확장 학습 (선택 / 과제)

### 더 해볼 만한 것
- **재무제표 통합**: DART OpenAPI (https://opendart.fss.or.kr) 연동해 매출/영업이익 추세 추가
- **다국어 보고서**: 같은 데이터로 영문/국문 동시 생성
- **포트폴리오 비교**: 여러 종목을 한 번에 분석해 비교표 생성
- **알림 시스템**: 특정 조건(예: -5% 하락)에서 보고서 자동 생성

### 국방 도메인 응용 아이디어
오늘 패턴 — **다중 소스 → AI 분석 → 검증 가능한 보고서** — 을 다음에 응용해 보세요.

- 방산 산업 동향 모니터링 봇 (OSINT 기반)
- 공개 정보 기반 일일 정보 브리핑 자동화
- 정비/물자 데이터 기반 운영 보고 자동 생성
- 회의 녹취 + 작전 데이터 융합 결심보고서 초안

### 보안 체크리스트 ✅
- [ ] API 키를 코드에 하드코딩하지 않았는가?
- [ ] Anthropic Console에서 사용 한도(spending limit)를 설정했는가?
- [ ] 외부 LLM에 전송하면 안 되는 데이터를 입력하지 않았는가?
- [ ] 노트북을 다른 사람과 공유할 때 출력 셀에 민감 정보가 남아있지 않은가?
- [ ] Gradio `share=True` 링크가 만료되었거나 공유 종료되었는가?

---

🪖 **수고하셨습니다.** 다음 시간에는 오늘 만든 패턴을 군 환경 시나리오에 적용해 봅니다.